In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import time

def predictive_performance_estimate(data_estimator, graphs, targets, n_rep=3):
    start = time.time()
    scores = []
    mistakes_per_run = []
    for i in range(n_rep):
        train_graphs, test_graphs, train_targets, test_targets = train_test_split(graphs, targets, train_size=.7, random_state=i)
        data_estimator.fit(train_graphs, train_targets)
        preds = data_estimator.predict_proba(test_graphs)
        preds = preds[:,-1]
        score = roc_auc_score(test_targets, preds)
        scores.append(score)
        # Compute number of mistakes using a 0.5 threshold on positive class probability
        y_hat = (preds >= 0.5).astype(int)
        mistakes = int(np.sum(y_hat != np.asarray(test_targets)))
        mistakes_per_run.append(mistakes)
    mean, std = np.mean(scores), np.std(scores)
    end = time.time()
    elapsed = (end - start)/n_rep
    avg_mistakes = float(np.mean(mistakes_per_run)) if mistakes_per_run else 0.0
    return scores, mean, std, elapsed, avg_mistakes

from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph.vectorize import AbstractGraphTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

def get_estimator(nbits, decomposition_function, estimator_type='linear'):
    if estimator_type == 'linear':
        estimator = LogisticRegression(
            solver="saga",        # robust for large/sparse
            penalty="l2",
            class_weight="balanced",
            max_iter=5000,
            n_jobs=-1,
            random_state=0
        )
    else:
        estimator = ExtraTreesClassifier(
            n_estimators=300, n_jobs=-1, random_state=0
        )
    graph_vectorizer = AbstractGraphTransformer(
        nbits=nbits,
        decomposition_function=decomposition_function,
        return_dense=True,
        n_jobs=-1,
    )
    return GraphEstimator(transformer=graph_vectorizer, estimator=estimator)


In [ ]:
import abstractgraph_ml.topk as ag_topk

def compute_topk_curve(
    graphs,
    targets,
    df,
    label='',
    *,
    nbits=14,
    top_ks=(1, 2, 4, 8, 16, 32, 64),
    n_rep=10,
    estimator_type='nlinear',
):
    """Helper that runs the Top-K ranking pipeline via abstractgraph_ml.topk."""
    graph_estimator = get_estimator(
        nbits=nbits,
        decomposition_function=df,
        estimator_type=estimator_type,
    )
    topk_dfs, ranked_feature_ids, diagnostics = ag_topk.make_topk_df(
        graphs,
        targets,
        graph_estimator=graph_estimator,
        top_ks=top_ks,
        n_splits=5,
        use_permutation=False,
        random_state=42,
    )
    results = ag_topk.compute_topk_roc_results(
        graphs,
        targets,
        topk_dfs=topk_dfs,
        top_ks=top_ks,
        graph_estimator=graph_estimator,
        performance_fn=predictive_performance_estimate,
        performance_kwargs={'n_rep': n_rep},
    )
    ax = ag_topk.plot_topk_roc_curve(results, label=label)
    return ax, results


In [ ]:
from collections import Counter
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def count_instances_per_class(train_targets, test_targets, class_names=None, visualize=False):
    """
    Counts and displays the number of instances per class in training and testing datasets.
    
    Parameters:
    - train_targets (list or array-like): Class labels for the training set.
    - test_targets (list or array-like): Class labels for the testing set.
    - class_names (list, optional): List of class names. If None, class labels will be used.
    - visualize (bool, optional): If True, displays a bar chart with counts. Default is False.
    
    Returns:
    - train_counts (dict): Dictionary with class labels/names as keys and their counts in training set as values.
    - test_counts (dict): Dictionary with class labels/names as keys and their counts in testing set as values.
    """
    
    # Count occurrences in training and testing targets
    train_counter = Counter(train_targets)
    test_counter = Counter(test_targets)
    
    # If class names are provided, map them accordingly
    if class_names:
        train_counts = {class_name: train_counter.get(class_name, 0) for class_name in class_names}
        test_counts = {class_name: test_counter.get(class_name, 0) for class_name in class_names}
    else:
        # Use the sorted unique class labels from both train and test sets
        unique_classes = sorted(set(train_targets) | set(test_targets))
        train_counts = {cls: train_counter.get(cls, 0) for cls in unique_classes}
        test_counts = {cls: test_counter.get(cls, 0) for cls in unique_classes}
    
    # Display the counts
    print("Number of instances per class in the Training Set:")
    for cls, count in train_counts.items():
        print(f"  Class '{cls}': {count} instances")
    
    print("\nNumber of instances per class in the Testing Set:")
    for cls, count in test_counts.items():
        print(f"  Class '{cls}': {count} instances")
    
    # Optional: Visualize the counts using bar charts
    if visualize:
        # Prepare data for visualization
        data = []
        for cls in train_counts:
            data.append({
                'Class': cls,
                'Dataset': 'Training',
                'Count': train_counts[cls]
            })
        for cls in test_counts:
            data.append({
                'Class': cls,
                'Dataset': 'Testing',
                'Count': test_counts[cls]
            })
        df = pd.DataFrame(data)
        
        # Create a bar plot using seaborn
        sns.set(style="whitegrid")  # Set the style for the plot
        plt.figure(figsize=(10, 6))  # Define the figure size
        
        # Create the barplot with 'Class' on the x-axis, 'Count' on the y-axis, and 'Dataset' as hue
        bar_plot = sns.barplot(x='Class', y='Count', hue='Dataset', data=df)
        
        # Set the title of the plot
        plt.title('Number of Instances per Class in Training and Testing Sets')
        
        # Iterate over each bar in the plot to add the count labels on top
        for p in bar_plot.patches:
            height = p.get_height()  # Get the height of the bar (count value)
            # Annotate the bar with the count value, slightly above the bar
            bar_plot.annotate(f'{int(height)}',
                              (p.get_x() + p.get_width() / 2., height),
                              ha='center', va='bottom', 
                              fontsize=10, color='black', 
                              xytext=(0, 5),  # 5 points vertical offset
                              textcoords='offset points')
        
        # Adjust layout for better spacing
        plt.tight_layout()
        
        # Display the plot
        plt.show()
    
    return train_counts, test_counts


In [ ]:
# Revised split_graphs using abstractgraph pipelines (callable or XML).
from typing import Iterable, Tuple, Union, List
import networkx as nx

import abstractgraph.operators as ag_ops
from abstractgraph.graphs import AbstractGraph, graph_to_abstract_graph
from abstractgraph.labels import graph_hash_label_function_factory
from abstractgraph.xml import (
    register_from_module,
    operator_from_xml_string,
)

try:
    from abstractgraph.hashing import hash_graph
except Exception:
    hash_graph = None

DecompositionSpec = Union[str, callable]  # XML string or callable op

def split_graphs(
    graphs: Iterable[nx.Graph],
    target_feature: Union[int, nx.Graph],
    decomposition: DecompositionSpec,
    nbits: int,
) -> Tuple[List[nx.Graph], List[nx.Graph]]:
    """
    Split graphs into two lists based on whether their AbstractGraph decomposition contains target_feature.

    Args:
      graphs: Iterable of base NetworkX graphs.
      target_feature:
        - If int: treated as the target interpretation-node label (0..2**nbits-1).
        - If nx.Graph: hashed with `hash_graph(..., nbits)` to derive the target label.
      decomposition:
        - Callable: a pipeline function op(ag) -> ag built via abstractgraph.operators
        - str (XML): serialized pipeline; will be deserialized after registering ag_ops.
      nbits: Number of bits for the label hash space (2**nbits features).

    Returns:
      (pos_graphs, neg_graphs): graphs where target label is present/absent in the interpretation graph.
    """
    if isinstance(decomposition, str):
        register_from_module(ag_ops)
        decomposition_op = operator_from_xml_string(decomposition)
    elif callable(decomposition):
        decomposition_op = decomposition
    else:
        raise TypeError("decomposition must be a callable or an XML string")

    if isinstance(target_feature, int):
        target_label = target_feature
    elif isinstance(target_feature, nx.Graph) and hash_graph is not None:
        target_label = hash_graph(target_feature, nbits=nbits)
    else:
        raise ValueError(
            "target_feature must be an int label or an nx.Graph (and hash_graph must be available)"
        )

    pos_graphs, neg_graphs = [], []

    for G in graphs:
        ag = graph_to_abstract_graph(G, decomposition_function=decomposition_op, nbits=nbits)
        labels = [data.get("label") for _, data in ag.interpretation_graph.nodes(data=True)]
        if target_label in labels:
            pos_graphs.append(G)
        else:
            neg_graphs.append(G)

    return pos_graphs, neg_graphs


In [ ]:
from abstractgraph.graphs import AbstractGraph, graph_to_abstract_graph
from abstractgraph.display import display, display_graph, display_mappings, display_decomposition_graph, decomposition_to_graph
from abstractgraph.labels import graph_hash_label_function_factory
from abstractgraph.operators import *

def draw(graph, df, nbits=10):
    display_decomposition_graph(df)
    ag = graph_to_abstract_graph(graph, decomposition_function=df, nbits=nbits)
    display(ag, size=(12,6))
    display_mappings(ag, n_elements_per_row=10)

---

In [ ]:
target_df = compose(filter_by_number_of_nodes(number_of_nodes=(6)), filter_by_node_label(must_have_one_of=['N']), cycle())
nbits = 12

In [ ]:
from abstractgraph_graphicalizer.chem import PubChemLoader

loader = PubChemLoader(on_error="skip")
assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_id = assay_ids[3]
original_graphs, original_targets = loader.load(assay_id, limit_active=500, limit_inactive=500)
print(f'#graphs: {len(original_graphs)}')

In [ ]:
import random
from abstractgraph_graphicalizer.chem import draw_molecules
graph_id = random.randint(0, len(original_graphs)-1)
graph = original_graphs[graph_id]
draw_molecules([graph])
draw(graph, target_df, nbits=nbits)

In [ ]:
feature_id = 1231 #5 ring with N
feature_id = 3434 #6 ring with N

from abstractgraph.vectorize import AbstractGraphTransformer
# Build a transformer that applies the pipeline `df` and vectorizes each graph
target_graph_vectorizer = AbstractGraphTransformer(nbits=nbits, decomposition_function=target_df, return_dense=True, n_jobs=-1)
X = target_graph_vectorizer.fit_transform(original_graphs)


# Column-wise presence mask for the target feature id
# Supports both dense (np.ndarray) and sparse (CSR) outputs
_col = X[:, feature_id]
if hasattr(_col, 'toarray'):
    col_vals = _col.toarray().ravel()
else:
    col_vals = _col.ravel()
has_mask = (col_vals > 0).astype(int)


has_feature_id_graphs = [graph for graph, has_mask_flag in zip(original_graphs, has_mask) if has_mask_flag]
has_not_feature_id_graphs = [graph for graph, has_mask_flag in zip(original_graphs, has_mask) if not has_mask_flag]

print(np.sum(has_mask))
draw_molecules(has_feature_id_graphs[:7])
print('-'*100)
print(len(original_graphs)-np.sum(has_mask))
draw_molecules(has_not_feature_id_graphs[:7])

graphs = has_feature_id_graphs+has_not_feature_id_graphs
targets = [0]*len(has_feature_id_graphs)+[1]*len(has_not_feature_id_graphs)

In [ ]:
%%time
#df = graphlet(radius=3, number_of_nodes=(4,6))
#df = add(cycle(), tree())
#df = neighborhood(radius=(0,2))
df = compose_product(binary_combination(distance=0), cycle(), compose(neighborhood(radius=2), tree()))

from abstractgraph.vectorize import AbstractGraphTransformer
graph_vectorizer = AbstractGraphTransformer(
    nbits=nbits, 
    decomposition_function=df, 
    return_dense=True, 
    n_jobs=-1)
data_estimator = get_estimator(graph_vectorizer, estimator_type='nlinear')
scores, mean, std, elapsed, avg_mistakes = predictive_performance_estimate(data_estimator, graphs, targets, n_rep=2)
print(f"ROC-AUC: {mean:.3f}±{std:.3f}   #errs:{avg_mistakes}   {elapsed:.1f}s  [{elapsed/60:.1f}m]")

In [ ]:
%%time
top_ks = (1, 2, 4, 8, 16, 32, 64)
graph_estimator = get_estimator(
    nbits=nbits,
    decomposition_function=df,
    estimator_type='nlinear',
)
topk_dfs, ranked_feature_ids, diagnostics = ag_topk.make_topk_df(
    graphs,
    targets,
    graph_estimator=graph_estimator,
    top_ks=top_ks,
    n_splits=5,
    use_permutation=False,
    random_state=42,
)


In [ ]:
import random
n_graphs = 6
n_graphs_per_line = n_graphs
rng = random.Random()
picked = rng.sample(graphs, min(n_graphs, len(graphs)))
draw_molecules(picked, n_graphs_per_line=n_graphs_per_line)

import abstractgraph_ml.importance as ag_importance
fig, axes = ag_importance.plot_graph_node_saliency_grid(
    graphs=picked,
    decomposition_function=df,
    ranked_feature_ids=ranked_feature_ids,
    nbits=nbits,
    n_elements_per_row=n_graphs_per_line,
    figsize_per_graph=(3.5, 3.5),
    cmap='YlOrRd',
    width_range=(1.5, 8.5),
    color_value_range=(0.2, 1.0),
    node_agg='max',
    edge_stat='mean',
)


In [ ]:
G = picked[0]
topk_df = topk_dfs[3]
draw_molecules([G])
draw(G, topk_df)
draw(G, compose(intersection(node_size=(5,100)), topk_df))

In [ ]:
%%time
results = ag_topk.compute_topk_roc_results(
    graphs,
    targets,
    topk_dfs=topk_dfs,
    top_ks=top_ks,
    graph_estimator=graph_estimator,
    performance_fn=predictive_performance_estimate,
    performance_kwargs={'n_rep': 5},
)

ax = ag_topk.plot_topk_roc_curve(results)


---

In [ ]:
df = graphlet(radius=2, number_of_nodes=(3,4))
display_decomposition_graph(df)
ax, results_list = compute_topk_curve(graphs, targets, df, label='graphlet')


In [ ]:
df = neighborhood(radius=(0,2))
display_decomposition_graph(df)
ax, results_list = compute_topk_curve(graphs, targets, df, label='neighborhood')


In [ ]:
df = compose_product(binary_combination(distance=0), cycle(), compose(neighborhood(radius=2), tree()))
display_decomposition_graph(df)
ax, results_list = compute_topk_curve(graphs, targets, df, label='cycle')


In [ ]:
%%time
dfs = [graphlet(radius=2, number_of_nodes=(3,4)),
       neighborhood(radius=(0,2)),
       compose_product(binary_combination(distance=0), cycle(), compose(neighborhood(radius=2), tree()))
       ]
labels = ['Graphlet r=2 (3-4 nodes)', 'Neighborhood r=0-2', 'Cycle+Tree+Product']
graph_estimator = get_estimator(
    nbits=14,
    decomposition_function=dfs[0],
    estimator_type='nlinear',
)
ax, results_list = ag_topk.plot_topk_roc_curves(
    graphs,
    targets,
    dfs=dfs,
    labels=labels,
    graph_estimator=graph_estimator,
    top_ks=(1, 2, 4, 8, 16, 32),
    performance_fn=predictive_performance_estimate,
    performance_kwargs={'n_rep': 30},
)
ax.figure.savefig('topk_roc_auc_comparison.svg')
